In [ ]:
import pandas as pd

# Read and display sample_submission.csv
sample_submission_df = pd.read_csv('/content/sample_submission.csv')
# Read and display /content/test.csv
test_df = pd.read_csv('/content/test.csv')
# Read and display /content/train.csv
train_df = pd.read_csv('/content/train.csv')


In [ ]:
print(sample_submission_df.dtypes)

ID             int64
Prediction    object
dtype: object


In [ ]:
print(train_df.dtypes)

id                 int64
prompt            object
A                 object
B                 object
C                 object
D                 object
E                 object
answer            object
prompt_cleaned    object
prompt_tokens     object
A_cleaned         object
A_tokens          object
B_cleaned         object
B_tokens          object
C_cleaned         object
C_tokens          object
D_cleaned         object
D_tokens          object
E_cleaned         object
E_tokens          object
prompt_tfidf      object
A_tfidf           object
B_tfidf           object
C_tfidf           object
D_tfidf           object
E_tfidf           object
prompt_w2v        object
A_w2v             object
B_w2v             object
C_w2v             object
D_w2v             object
E_w2v             object
dtype: object


In [ ]:
print(test_df.dtypes)

id                 int64
prompt            object
A                 object
B                 object
C                 object
D                 object
E                 object
prompt_cleaned    object
prompt_tokens     object
A_cleaned         object
A_tokens          object
B_cleaned         object
B_tokens          object
C_cleaned         object
C_tokens          object
D_cleaned         object
D_tokens          object
E_cleaned         object
E_tokens          object
prompt_tfidf      object
A_tfidf           object
B_tfidf           object
C_tfidf           object
D_tfidf           object
E_tfidf           object
prompt_w2v        object
A_w2v             object
B_w2v             object
C_w2v             object
D_w2v             object
E_w2v             object
dtype: object


In [ ]:
sample_submission_df.head()

,ID,Prediction
0,1,A B C
1,2,A B C
2,3,A B C
3,4,A B C
4,5,A B C


In [ ]:
test_df.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [ ]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


# **MILESTONE 1 STARTS HERE**

In [ ]:
# MILESTONE 1 STARTS HERE

In [ ]:
# Section 1: Install & Import Libraries
!pip install gensim

import re
import string
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from gensim.models import Word2Vec

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

True

In [ ]:
# Section 2: Text Cleaning, Tokenization & Missing Data Handling

def clean_text(text):

    # Handle missing values
    if not isinstance(text, str):
        text = ''

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = re.sub(
        f'[{re.escape(string.punctuation)}]',
        '',
        text
    )

    # Tokenization
    tokens = word_tokenize(text)

    # Stopword removal
    stop_words = set(stopwords.words('english'))

    filtered_tokens = [
        word
        for word in tokens
        if word not in stop_words and word.isalpha()
    ]

    return ' '.join(filtered_tokens), filtered_tokens

In [ ]:
import nltk

# Download required NLTK resources
resources = [
    "punkt",
    "punkt_tab",
    "stopwords",
    "wordnet",
    "omw-1.4"
]

for resource in resources:
    nltk.download(resource)

# Section 3: Apply Cleaning to Train and Test Data
text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']

print("Cleaning text...")

for df in [train_df, test_df]:
    for col in text_cols:
        df[col] = df[col].fillna('')

        cleaned = df[col].apply(clean_text)

        df[f'{col}_cleaned'] = cleaned.apply(lambda x: x[0])
        df[f'{col}_tokens'] = cleaned.apply(lambda x: x[1])

print("Cleaning completed.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Cleaning text...
Cleaning completed.


In [ ]:
# Section 4: Generate TF-IDF Embeddings

print("Generating TF-IDF embeddings...")

all_cleaned_text = []

for col in [f'{c}_cleaned' for c in text_cols]:

    all_cleaned_text.extend(
        train_df[col].tolist()
    )

# Fit only on training data
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000
)

tfidf_vectorizer.fit(all_cleaned_text)

# Train data embeddings
train_df['prompt_tfidf'] = list(
    tfidf_vectorizer.transform(
        train_df['prompt_cleaned']
    )
)

for col in ['A', 'B', 'C', 'D', 'E']:

    train_df[f'{col}_tfidf'] = list(
        tfidf_vectorizer.transform(
            train_df[f'{col}_cleaned']
        )
    )

# Test data embeddings
test_df['prompt_tfidf'] = list(
    tfidf_vectorizer.transform(
        test_df['prompt_cleaned']
    )
)

for col in ['A', 'B', 'C', 'D', 'E']:

    test_df[f'{col}_tfidf'] = list(
        tfidf_vectorizer.transform(
            test_df[f'{col}_cleaned']
        )
    )

print("TF-IDF completed.")

Generating TF-IDF embeddings...
TF-IDF completed.


In [ ]:
# Section 5: Generate Word2Vec Embeddings

print("Training Word2Vec model...")

all_tokens = []

for col in [f'{c}_tokens' for c in text_cols]:

    all_tokens.extend(
        train_df[col].tolist()
    )

word2vec_model = Word2Vec(
    sentences=all_tokens,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

print("Word2Vec training completed.")

Training Word2Vec model...
Word2Vec training completed.


In [ ]:
# Section 6: Convert Sentences to Word2Vec Embeddings
def get_word2vec_embedding(
    tokens,
    model,
    vector_size=100
):

    vectors = []

    for token in tokens:

        if token in model.wv:
            vectors.append(
                model.wv[token]
            )

    if len(vectors) > 0:

        return np.mean(
            vectors,
            axis=0
        )

    return np.zeros(vector_size)

In [ ]:
# Section 7: Apply Word2Vec Embeddings
train_df['prompt_w2v'] = train_df[
    'prompt_tokens'
].apply(
    lambda x:
    get_word2vec_embedding(
        x,
        word2vec_model
    )
)

for col in ['A', 'B', 'C', 'D', 'E']:

    train_df[f'{col}_w2v'] = train_df[
        f'{col}_tokens'
    ].apply(
        lambda x:
        get_word2vec_embedding(
            x,
            word2vec_model
        )
    )

test_df['prompt_w2v'] = test_df[
    'prompt_tokens'
].apply(
    lambda x:
    get_word2vec_embedding(
        x,
        word2vec_model
    )
)

for col in ['A', 'B', 'C', 'D', 'E']:

    test_df[f'{col}_w2v'] = test_df[
        f'{col}_tokens'
    ].apply(
        lambda x:
        get_word2vec_embedding(
            x,
            word2vec_model
        )
    )

print("Word2Vec embeddings generated.")

Word2Vec embeddings generated.


In [ ]:
# Section 8: Cosine Similarity Function
def compute_cosine_similarity(
    vec1,
    vec2
):

    # TF-IDF sparse matrices
    if hasattr(vec1, "toarray"):

        return cosine_similarity(
            vec1,
            vec2
        )[0][0]

    # Word2Vec vectors
    return cosine_similarity(
        vec1.reshape(1, -1),
        vec2.reshape(1, -1)
    )[0][0]

In [ ]:
# Section 9: Example Similarity Calculation
print("\nExample Similarities")

tfidf_similarity = compute_cosine_similarity(
    train_df['prompt_tfidf'].iloc[0],
    train_df['A_tfidf'].iloc[0]
)

print(
    f"TF-IDF Similarity: {tfidf_similarity:.4f}"
)

w2v_similarity = compute_cosine_similarity(
    train_df['prompt_w2v'].iloc[0],
    train_df['A_w2v'].iloc[0]
)

print(
    f"Word2Vec Similarity: {w2v_similarity:.4f}"
)


Example Similarities
TF-IDF Similarity: 0.1596
Word2Vec Similarity: 0.7651


In [ ]:
# Section 10: AP@3 Calculation
def calculate_ap_at_k(
    actual,
    predicted,
    k=3
):

    predicted = predicted[:k]

    for i, p in enumerate(predicted):

        if p == actual:

            return 1.0 / (i + 1)

    return 0.0

In [ ]:
# Section 11: mAP@3 Calculation
def calculate_map_k(
    df,
    embedding_type='tfidf',
    k=3
):

    options = [
        'A',
        'B',
        'C',
        'D',
        'E'
    ]

    ap_scores = []

    for _, row in df.iterrows():

        prompt_vec = row[
            f'prompt_{embedding_type}'
        ]

        actual_answer = row['answer']

        similarities = []

        for opt in options:

            option_vec = row[
                f'{opt}_{embedding_type}'
            ]

            similarity = compute_cosine_similarity(
                prompt_vec,
                option_vec
            )

            similarities.append(
                (
                    opt,
                    similarity
                )
            )

        similarities.sort(
            key=lambda x: x[1],
            reverse=True
        )

        predicted_labels = [
            opt
            for opt, sim
            in similarities
        ]

        ap = calculate_ap_at_k(
            actual_answer,
            predicted_labels,
            k
        )

        ap_scores.append(ap)

    return np.mean(ap_scores)

In [ ]:
# Section 12: Calculate Final mAP@3 Scores
print("\nCalculating mAP@3")

map3_tfidf = calculate_map_k(
    train_df,
    embedding_type='tfidf',
    k=3
)

print(
    f"TF-IDF mAP@3: {map3_tfidf:.4f}"
)

map3_w2v = calculate_map_k(
    train_df,
    embedding_type='w2v',
    k=3
)

print(
    f"Word2Vec mAP@3: {map3_w2v:.4f}"
)


Calculating mAP@3
TF-IDF mAP@3: 0.2871
Word2Vec mAP@3: 0.3777


In [ ]:
# Section 13: Concept Summary
print("""
NLP Tasks Completed:

1. Text Cleaning
   - Lowercasing
   - Punctuation Removal
   - Stopword Removal

2. Tokenization
   - Split text into words

3. Missing Data Handling
   - Replaced NaN with empty strings

4. TF-IDF Embeddings
   - Frequency-based representation

5. Word2Vec Embeddings
   - Semantic representation

6. Cosine Similarity
   - Measures similarity between prompt and options

7. mAP@3
   - Evaluates ranking quality
   - Correct answer at:
       Rank 1 -> 1.0
       Rank 2 -> 0.5
       Rank 3 -> 0.333
       Outside Top 3 -> 0
""")


NLP Tasks Completed:

1. Text Cleaning
   - Lowercasing
   - Punctuation Removal
   - Stopword Removal

2. Tokenization
   - Split text into words

3. Missing Data Handling
   - Replaced NaN with empty strings

4. TF-IDF Embeddings
   - Frequency-based representation

5. Word2Vec Embeddings
   - Semantic representation

6. Cosine Similarity
   - Measures similarity between prompt and options

7. mAP@3
   - Evaluates ranking quality
   - Correct answer at:
       Rank 1 -> 1.0
       Rank 2 -> 0.5
       Rank 3 -> 0.333
       Outside Top 3 -> 0



# **MILESTONE 2 STARTS HERE**

In [ ]:
# MILESTONE 2 STARTS HERE

In [ ]:
!pip install -q transformers datasets sentence-transformers accelerate

from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer
import torch

In [ ]:
# Step 1: Load the model
# Load a lightweight pre-trained transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
# Step 2: Test on one sentence

sentence = "The Earth revolves around the Sun."

embedding = model.encode(sentence)

print("Embedding shape:", embedding.shape)
print("First 10 values:")
print(embedding[:10])

Embedding shape: (384,)
First 10 values:
[ 0.03163033  0.05801901  0.07984006  0.04298688  0.04338428 -0.0028553
 -0.00184029 -0.02855255  0.10804769 -0.02113786]


In [ ]:
# Step 3: Generate embeddings for your dataset

text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']

print("Generating transformer embeddings...")

for df in [train_df, test_df]:
    for col in text_cols:
        df[f'{col}_bert'] = df[col].fillna('').apply(model.encode)

print("Done!")

Generating transformer embeddings...
Done!


In [ ]:
# Step 4: Check the new columns
print(train_df.columns)
# Step 5: Verify one embedding
print(train_df['prompt_bert'].iloc[0].shape)

Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'prompt_cleaned',
       'prompt_tokens', 'A_cleaned', 'A_tokens', 'B_cleaned', 'B_tokens',
       'C_cleaned', 'C_tokens', 'D_cleaned', 'D_tokens', 'E_cleaned',
       'E_tokens', 'prompt_tfidf', 'A_tfidf', 'B_tfidf', 'C_tfidf', 'D_tfidf',
       'E_tfidf', 'prompt_w2v', 'A_w2v', 'B_w2v', 'C_w2v', 'D_w2v', 'E_w2v',
       'prompt_bert', 'A_bert', 'B_bert', 'C_bert', 'D_bert', 'E_bert'],
      dtype='object')
(384,)


In [ ]:
# Compute cosine similarity

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

option_cols = ['A', 'B', 'C', 'D', 'E']

def rank_options(row):
    prompt = row['prompt_bert']

    scores = {}

    for option in option_cols:
        sim = cosine_similarity(
            [prompt],
            [row[f'{option}_bert']]
        )[0][0]

        scores[option] = sim

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return (
        [x[0] for x in ranked],      # Ranked options
        [x[1] for x in ranked]       # Similarity scores
    )

train_df['bert_rank'], train_df['bert_scores'] = zip(
    *train_df.apply(rank_options, axis=1)
)

In [ ]:
print(train_df[['prompt', 'answer', 'bert_rank', 'bert_scores']].head())

                                              prompt answer        bert_rank  \
0  Pick the best possible answer: What is Martin ...      B  [C, D, B, E, A]   
1        What is accelerator-based light-ion fusion?      A  [B, C, D, A, E]   
2  Determine the correct option: What is the term...      C  [B, A, D, C, E]   
3  Select the most accurate option: What is Marti...      B  [C, D, B, E, A]   
4  Identify the correct statement: What is the co...      A  [E, D, A, B, C]   

                                         bert_scores  
0  [0.79401636, 0.7700757, 0.76580983, 0.73231304...  
1  [0.8831366, 0.8803911, 0.86325777, 0.8588867, ...  
2  [0.27362654, 0.20180833, 0.13595162, 0.0733071...  
3  [0.81603515, 0.7933612, 0.77533984, 0.75673115...  
4  [0.7421836, 0.73380864, 0.63698304, 0.6320634,...  


In [ ]:
# Predict the Top 3 answers

train_df['bert_prediction'] = train_df['bert_rank'].apply(lambda x: x[:3])

print(train_df[['answer', 'bert_prediction']].head())

  answer bert_prediction
0      B       [C, D, B]
1      A       [B, C, D]
2      C       [B, A, D]
3      B       [C, D, B]
4      A       [E, D, A]


In [ ]:
# Evaluate using mAP@3

# Since you've already implemented mAP@3, you can reuse it:

import numpy as np

def apk(actual, predicted, k=3):
    """
    Computes the Average Precision at k.
    """
    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0

    for i, p in enumerate(predicted):
        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score


def mapk(actuals, predictions, k=3):
    """
    Computes the Mean Average Precision at k.
    """
    return np.mean([
        apk(a, p, k)
        for a, p in zip(actuals, predictions)
    ])

score = mapk(
    train_df['answer'].tolist(),
    train_df['bert_prediction'].tolist(),
    k=3
)

print(f"Transformer mAP@3: {score:.4f}")

Transformer mAP@3: 0.4231


In [ ]:
# Step 1: Load a BERT tokenizer

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Tokenizer loaded!")

Tokenizer loaded!


In [ ]:
# Step 2: Tokenize one sentence

text = train_df.loc[0, "prompt"]

tokens = tokenizer(text)

print(tokens)

{'input_ids': [101, 4060, 1996, 2190, 2825, 3437, 1024, 2054, 2003, 3235, 2002, 5178, 13327, 1005, 1055, 3193, 2006, 1996, 3276, 2090, 2051, 1998, 2529, 4598, 1029, 2426, 1996, 3205, 7047, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
# Step 3: Decode the tokens

token_ids = tokens["input_ids"]

print(tokenizer.convert_ids_to_tokens(token_ids))

['[CLS]', 'pick', 'the', 'best', 'possible', 'answer', ':', 'what', 'is', 'martin', 'he', '##ide', '##gger', "'", 's', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', '?', 'among', 'the', 'listed', 'options', '.', '[SEP]']


In [ ]:
# Part 2: Load a BERT Model

from transformers import AutoModel

bert = AutoModel.from_pretrained("bert-base-uncased")

print("Model loaded!")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded!


In [ ]:
# Generate embeddings

import torch

text = train_df.loc[0, "prompt"]

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

with torch.no_grad():
    outputs = bert(**inputs)

print(outputs.last_hidden_state.shape)

torch.Size([1, 31, 768])


In [ ]:
# Get the sentence embedding

# A common approach is to use the embedding of the [CLS] token:

cls_embedding = outputs.last_hidden_state[:, 0, :]

print(cls_embedding.shape)

torch.Size([1, 768])


In [ ]:
# Next: Zero-Shot Classification

# This is one of the coolest features of Hugging Face because you don't train anything.

from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

print("Zero-shot model loaded!")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Zero-shot model loaded!


In [ ]:
# Test for zero shot classification

text = train_df.loc[0, "prompt"]

labels = [
    "Science",
    "Mathematics",
    "History",
    "Geography",
    "Literature"
]

result = classifier(text, candidate_labels=labels)

print(result)

{'sequence': "Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.", 'labels': ['History', 'Science', 'Literature', 'Mathematics', 'Geography'], 'scores': [0.3105526864528656, 0.24705563485622406, 0.18557456135749817, 0.12907561659812927, 0.12774154543876648]}


In [ ]:
# Trying zero shot classification with better labels

text = train_df.loc[0, "prompt"]

labels = [
    "Philosophy",
    "Science",
    "Mathematics",
    "History",
    "Literature"
]

result = classifier(text, candidate_labels=labels)

print(result)

{'sequence': "Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.", 'labels': ['Philosophy', 'History', 'Science', 'Literature', 'Mathematics'], 'scores': [0.7273721098899841, 0.09706450253725052, 0.07721824198961258, 0.058002080768346786, 0.04034310579299927]}


In [ ]:
# Trying zero shot classification with options small version

row = train_df.iloc[0]

candidate_labels = [
    row["A"],
    row["B"],
    row["C"],
    row["D"],
    row["E"]
]

result = classifier(
    row["prompt"],
    candidate_labels=candidate_labels
)

print(result)

{'sequence': "Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.", 'labels': ['Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.', "Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.", 'Martin Heidegger believes that the relationship between time and human existence is cyclical. The past and present are interconnected and the future is predetermined. Human beings do not have free will.', 'Martin Heidegger does not believe in the existence

In [ ]:
# How to evaluate it on your entire dataset

# You can apply the same idea to every row and generate top-3 predictions:

option_cols = ['A', 'B', 'C', 'D', 'E']

def zero_shot_predict(row):
    labels = [row[col] for col in option_cols]

    result = classifier(
        row["prompt"],
        candidate_labels=labels
    )

    ranked = []

    for label in result["labels"]:
        ranked.append(option_cols[labels.index(label)])

    return ranked[:3]

train_df["zero_shot_prediction"] = train_df.apply(
    zero_shot_predict,
    axis=1
)

print(train_df[["answer", "zero_shot_prediction"]].head())

  answer zero_shot_prediction
0      B            [B, A, D]
1      A            [E, D, A]
2      C            [B, C, E]
3      B            [B, A, D]
4      A            [A, B, E]


In [ ]:
# mAP@3 evaluation

score = mapk(
    train_df["answer"].tolist(),
    train_df["zero_shot_prediction"].tolist(),
    k=3
)

print(f"Zero-Shot mAP@3: {score:.4f}")

Zero-Shot mAP@3: 0.5658


In [ ]:
# Topic 1: Softmax vs Sigmoid

# This is mostly about understanding how models produce probabilities.

# Softmax

# Used when exactly one answer is correct.
# Code for demonstration:

import torch
import torch.nn.functional as F

logits = torch.tensor([5.2,1.3,-0.8,-2.1])

softmax_probs = F.softmax(logits, dim=0)

print(softmax_probs)
print(softmax_probs.sum())

tensor([9.7714e-01, 1.9779e-02, 2.4221e-03, 6.6010e-04])
tensor(1.0000)


In [ ]:
# 2. Sigmoid

# Sigmoid when there is more than 1 correct answer.
# Code for demonstration:
import torch
import torch.nn.functional as F

logits = torch.tensor([5.2,1.3,-0.8,-2.1])

sigmoid_probs = torch.sigmoid(logits)

print(sigmoid_probs)

tensor([0.9945, 0.7858, 0.3100, 0.1091])


In [ ]:
# 3. Prompting an SLM

# This can be applied to your project.

# Instead of ranking embeddings or using zero-shot classification, you ask a language model to answer the question.

row = train_df.iloc[0]

prompt = f"""
Question:
{row['prompt']}

Options:

A. {row['A']}

B. {row['B']}

C. {row['C']}

D. {row['D']}

E. {row['E']}

Return only the letter of the correct answer.
"""

print(prompt)


Question:
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.

Options:

A. Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.

B. Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.

C. Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present.

D. Martin Heidegger believes that

In [ ]:
# MILESTONE 2 QUESTIONS STARTS HERE

In [ ]:
from datasets import load_dataset

# Load train.csv (without pandas)
dataset = load_dataset(
    "csv",
    data_files="/content/train.csv",
    split="train"
)

# Create the combined_text column using .map()
def combine_text(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

dataset = dataset.map(combine_text)

# Get the row at index 51
row_51 = dataset[51]

# Print the combined text
print(row_51["combined_text"])

# Print its character length
print("Character length:", len(row_51["combined_text"]))

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
Character length: 614


In [ ]:
from transformers import AutoTokenizer

# Initialize the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Print the vocabulary size
print("Vocabulary size:", tokenizer.vocab_size)

Vocabulary size: 30522


In [ ]:
from transformers import AutoTokenizer

# Load the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Get the [SEP] token ID
sep_token_id = tokenizer.sep_token_id

print("SEP Token:", tokenizer.sep_token)
print("SEP Token ID:", sep_token_id)

SEP Token: [SEP]
SEP Token ID: 102


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/content/train.csv",
    split="train"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Convert every prompt to a string
prompts = [str(x) for x in dataset["prompt"]]

# Tokenize
encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Shape
print(encoded["input_ids"].shape)

torch.Size([2000, 128])


In [ ]:
from transformers import AutoModel

# Load pretrained BERT model
model = AutoModel.from_pretrained("bert-base-uncased")

# Get configuration
hidden_size = model.config.hidden_size
num_attention_heads = model.config.num_attention_heads

# Calculate head dimension
head_dimension = hidden_size // num_attention_heads

print("Hidden Size:", hidden_size)
print("Attention Heads:", num_attention_heads)
print("Dimension per Head:", head_dimension)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hidden Size: 768
Attention Heads: 12
Dimension per Head: 64


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
import torch

# Load the dataset
dataset = load_dataset(
    "csv",
    data_files="/content/train.csv",
    split="train"
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Get the prompt from row 0
text = dataset[0]["prompt"]

# Tokenize using DEFAULT settings
inputs = tokenizer(text, return_tensors="pt")

# Pass through the model
with torch.no_grad():
    outputs = model(**inputs)

# Print the shape of the last hidden state
print("last_hidden_state shape:", outputs.last_hidden_state.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


last_hidden_state shape: torch.Size([1, 31, 768])


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
import torch

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/content/train.csv",
    split="train"
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Get prompt from row 0
text = dataset[0]["prompt"]

# Tokenize
inputs = tokenizer(text, return_tensors="pt")

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# Extract [CLS] embedding
cls_embedding = outputs.last_hidden_state[0, 0, :]

# Sum first 5 values
answer = cls_embedding[:5].sum().item()

print("Answer:", round(answer, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer: -1.2001


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

# Input sentence
text = "Light-ion fusion is a technique."

# Tokenize
inputs = tokenizer(text, return_tensors="pt")

# Show tokens and their indices
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Tokens:")
for i, token in enumerate(tokens):
    print(f"{i}: {token}")

# Find the index of "fusion"
fusion_index = tokens.index("fusion")
print("\nFusion token index:", fusion_index)

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# Last layer attention
last_layer_attention = outputs.attentions[-1]

# First attention head
head0_attention = last_layer_attention[0, 0]

# Attention from [CLS] (token 0) to "fusion"
attention_weight = head0_attention[0, fusion_index].item()

print("\nAttention Weight:", round(attention_weight, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens:
0: [CLS]
1: light
2: -
3: ion
4: fusion
5: is
6: a
7: technique
8: .
9: [SEP]

Fusion token index: 4

Attention Weight: 0.1025


In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

# Load train.csv using Hugging Face datasets
dataset = load_dataset(
    "csv",
    data_files="/content/train.csv",
    split="train"
)

# Load the Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Get row 0
prompt = dataset[0]["prompt"]
option_b = dataset[0]["B"]

# Generate embeddings
prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

# Compute cosine similarity
similarity = util.cos_sim(prompt_embedding, option_b_embedding)

# Print the similarity score
print("Cosine Similarity:", round(similarity.item(), 4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Cosine Similarity: 0.7658


In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# -----------------------------
# Load Dataset
# -----------------------------
dataset = load_dataset(
    "csv",
    data_files="/content/train.csv",
    split="train"
)

# -----------------------------
# mAP@3 functions
# -----------------------------
def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    for i, p in enumerate(predicted):
        if p == actual:
            return 1/(i+1)
    return 0.0

def mapk(actuals, predictions, k=3):
    return np.mean([apk(a,p,k) for a,p in zip(actuals,predictions)])

# -----------------------------
# MiniLM Model
# -----------------------------
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

option_letters = ["A","B","C","D","E"]

tfidf_predictions = []
minilm_predictions = []
answers = []

# -----------------------------
# Process every row
# -----------------------------
for row in dataset:

    prompt = row["prompt"]
    options = [row[o] for o in option_letters]
    answers.append(row["answer"])

    #############################################
    # Pipeline 1 : TF-IDF
    #############################################

    docs = [prompt] + options

    tfidf = TfidfVectorizer()
    X = tfidf.fit_transform(docs)

    prompt_vec = X[0]
    option_vecs = X[1:]

    scores = []

    for i in range(5):
        sim = (prompt_vec @ option_vecs[i].T).toarray()[0][0]
        scores.append(sim)

    ranking = np.argsort(scores)[::-1]

    tfidf_predictions.append(
        [option_letters[i] for i in ranking[:3]]
    )

    #############################################
    # Pipeline 2 : MiniLM
    #############################################

    prompt_emb = model.encode(prompt, convert_to_tensor=True)

    sims = []

    for option in options:
        emb = model.encode(option, convert_to_tensor=True)
        sims.append(
            util.cos_sim(prompt_emb, emb).item()
        )

    ranking = np.argsort(sims)[::-1]

    minilm_predictions.append(
        [option_letters[i] for i in ranking[:3]]
    )

# -----------------------------
# Question 1
# -----------------------------
minilm_map3 = mapk(
    answers,
    minilm_predictions,
    k=3
)

print("MiniLM MAP@3 =", round(minilm_map3,4))

# -----------------------------
# Question 2
# -----------------------------
count = 0

for actual, tfidf_pred, minilm_pred in zip(
    answers,
    tfidf_predictions,
    minilm_predictions
):

    if actual not in tfidf_pred and actual in minilm_pred:
        count += 1

print("Improved Questions =", count)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

MiniLM MAP@3 = 0.4231
Improved Questions = 574


In [ ]:
from datasets import load_dataset
from transformers import pipeline

# Load the dataset
dataset = load_dataset(
    "csv",
    data_files="/content/train.csv",
    split="train"
)

# Initialize the zero-shot classification pipeline
classifier = pipeline("zero-shot-classification")

# Get the prompt and options from row index 1
prompt = dataset[1]["prompt"]

candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"]
]

# Perform zero-shot classification
result = classifier(
    prompt,
    candidate_labels=candidate_labels
)

# Display the ranked labels and scores
for label, score in zip(result["labels"], result["scores"]):
    print(f"{score:.4f} -> {label}")

# Print the top-ranked probability
print("\nTop Probability:", round(result["scores"][0], 4))

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

0.4575 -> Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.
0.2751 -> Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.
0.2675 -> Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively dif

In [ ]:
from datasets import load_dataset
from transformers import pipeline

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/content/train.csv",
    split="train"
)

# Initialize zero-shot classifier
classifier = pipeline("zero-shot-classification")

# Prompt and candidate labels from row index 1
prompt = dataset[1]["prompt"]

candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"]
]

# -------------------------------
# Softmax (default)
# -------------------------------
result_softmax = classifier(
    prompt,
    candidate_labels=candidate_labels
)

softmax_sum = sum(result_softmax["scores"])

# -------------------------------
# Independent Sigmoids
# -------------------------------
result_sigmoid = classifier(
    prompt,
    candidate_labels=candidate_labels,
    multi_label=True
)

sigmoid_sum = sum(result_sigmoid["scores"])

# -------------------------------
# Absolute Difference
# -------------------------------
difference = abs(softmax_sum - sigmoid_sum)

print("Softmax scores:", result_softmax["scores"])
print("Softmax sum:", softmax_sum)

print("\nSigmoid scores:", result_sigmoid["scores"])
print("Sigmoid sum:", sigmoid_sum)

print("\nAbsolute Difference:", round(difference, 4))

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Softmax scores: [0.45745277404785156, 0.2750643789768219, 0.26748284697532654]
Softmax sum: 1.0

Sigmoid scores: [0.0004692690272349864, 2.063588544842787e-05, 1.9700582924997434e-05]
Sigmoid sum: 0.0005096054956084117

Absolute Difference: 0.9995


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/content/train.csv",
    split="train"
)

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# Row 0
row = dataset[0]

# Create the exact prompt
prompt = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} "
    f"or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

# Tokenize
inputs = tokenizer(prompt, return_tensors="pt")

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

# Decode
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Generated Answer:", answer)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generated Answer: B


# **MILESTONE 3 STARTS HERE**

In [ ]:
# MILESTONE 3 STARTS HERE

In [3]:
!pip install faiss-cpu #install FAISS

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('train.csv')

print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating knowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [4]:
zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

ans_150 = str(row_150[row_150["answer"]])

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [5]:
result = zs(
    prompt_150,
    candidate_labels=labels_150,
    multi_label=False
)

print(result)

{'sequence': 'Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.', 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."', 'The butterfly effect is the phenomenon that a small change in the initial conditio

In [6]:
print("Correct option letter:", row_150["answer"])
print("Correct answer text:", ans_150)

Correct option letter: C
Correct answer text: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."


In [7]:
# Embed the prompt
query_embedding = model.encode([prompt_150])

# Search the FAISS index for the top 10 nearest neighbors
distances, indices = index.search(query_embedding, k=10)

print("Top 10 retrieved indices:")
for rank, idx in enumerate(indices[0], start=1):
    print(f"Rank {rank}: KB Index = {idx}")

# Find the rank of the true document (originally at index 150)
true_index = 150

if true_index in indices[0]:
    rank = list(indices[0]).index(true_index) + 1
    print(f"\nTrue document found at Rank {rank}")
else:
    print("\nTrue document NOT found in the top 10.")

Top 10 retrieved indices:
Rank 1: KB Index = 663
Rank 2: KB Index = 1701
Rank 3: KB Index = 1269
Rank 4: KB Index = 1532
Rank 5: KB Index = 576
Rank 6: KB Index = 847
Rank 7: KB Index = 1693
Rank 8: KB Index = 1906
Rank 9: KB Index = 168
Rank 10: KB Index = 150

True document found at Rank 10


In [8]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

retrieved_indices = indices[0]

docs_10 = [kb[i] for i in retrieved_indices]

pairs = [[prompt_150, doc] for doc in docs_10]

ce_scores = cross_encoder.predict(pairs)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [9]:
# Pair each retrieved document with its score
results = list(zip(retrieved_indices, ce_scores))

# Sort by Cross-Encoder score (highest first)
results = sorted(results, key=lambda x: x[1], reverse=True)

print("Cross-Encoder Ranking:\n")

for rank, (idx, score) in enumerate(results, start=1):
    print(f"Rank {rank}: KB Index = {idx}, Score = {score:.4f}")

# Find where the true document (index 150) appears
for rank, (idx, score) in enumerate(results, start=1):
    if idx == 150:
        print(f"\n✅ True document is at Rank {rank}")
        break
else:
    print("\n❌ True document is not in the top 10.")

Cross-Encoder Ranking:

Rank 1: KB Index = 150, Score = 4.7585
Rank 2: KB Index = 847, Score = 4.7526
Rank 3: KB Index = 1693, Score = 4.7526
Rank 4: KB Index = 1906, Score = 4.7526
Rank 5: KB Index = 1269, Score = 4.7375
Rank 6: KB Index = 1532, Score = 4.7375
Rank 7: KB Index = 168, Score = 4.7072
Rank 8: KB Index = 576, Score = 4.6870
Rank 9: KB Index = 663, Score = 4.6602
Rank 10: KB Index = 1701, Score = 4.6602

✅ True document is at Rank 1


In [10]:
from transformers import AutoTokenizer

# Select row 42
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

# Embed the prompt
query_embedding = model.encode([prompt_42])

# Retrieve top 5 documents from FAISS
distances, indices = index.search(query_embedding, k=5)

retrieved_docs = [kb[i] for i in indices[0]]

# Concatenate documents with a single space
concatenated_docs = " ".join(retrieved_docs)

# Create the final input string
input_text = f"Context: {concatenated_docs} Question: {prompt_42}"

# Load the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize WITHOUT truncation
tokens = tokenizer(
    input_text,
    truncation=False,
    add_special_tokens=True
)

# Print the total number of tokens
print("Total tokens:", len(tokens["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Total tokens: 216


In [11]:
# Get the exact true document from the KB
true_document = kb[150]

# Create the RAG-augmented prompt
rag_prompt = f"Context: {true_document} Question: {prompt_150}"

# Run the zero-shot classifier
result = zs(
    rag_prompt,
    candidate_labels=labels_150,
    multi_label=False
)

# Display all labels and their scores
print("Labels and Scores:")
for label, score in zip(result["labels"], result["scores"]):
    print(f"{score:.3f} : {label}")

# Find the score assigned to the ground-truth answer
gt_score = dict(zip(result["labels"], result["scores"]))[ans_150]

print("\nGround-truth probability score:", round(gt_score, 3))

Labels and Scores:
0.989 : The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.004 : The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.003 : The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.002 : The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."
0.002 : The b

In [12]:
# Get an unrelated document from the KB
wrong_document = kb[999]

# Create the adversarial RAG prompt
adversarial_prompt = f"Context: {wrong_document} Question: {prompt_150}"

# Run the zero-shot classifier
result = zs(
    adversarial_prompt,
    candidate_labels=labels_150,
    multi_label=False
)

# Display all labels and scores
print("Labels and Scores:")
for label, score in zip(result["labels"], result["scores"]):
    print(f"{score:.3f} : {label}")

# Get the probability assigned to the ground-truth answer
gt_score = dict(zip(result["labels"], result["scores"]))[ans_150]

print("\nGround-truth probability score:", round(gt_score, 3))

Labels and Scores:
0.529 : The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.425 : The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.020 : The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.019 : The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical framework has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.007 : The butterfly effect is the phenomenon that a small change in the initial conditions of

In [13]:
# Evaluate Hit Rate@5 for the first 100 rows

hits = 0
total = 100

for i in range(total):
    row = train.iloc[i]

    # Get the prompt
    prompt = str(row["prompt"])

    # Get the correct answer text
    correct_letter = row["answer"]
    correct_text = str(row[correct_letter])

    # Embed the prompt
    query_embedding = model.encode([prompt])

    # Retrieve top 5 documents
    distances, indices = index.search(query_embedding, k=5)

    retrieved_docs = [kb[idx] for idx in indices[0]]

    # Check if the exact correct answer appears in any retrieved document
    if any(correct_text == doc for doc in retrieved_docs):
        hits += 1

# Compute Hit Rate
hit_rate = (hits / total) * 100

print(f"Hits: {hits}/{total}")
print(f"Hit Rate@5: {hit_rate:.1f}%")

Hits: 73/100
Hit Rate@5: 73.0%


In [14]:
from sentence_transformers import CrossEncoder
import numpy as np

# Load the Cross-Encoder (if not already loaded)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def map_at_3(predictions, true_answer):
    """
    predictions: list of predicted letters, e.g. ['C','A','E']
    true_answer: correct letter, e.g. 'C'
    """
    if true_answer in predictions:
        rank = predictions.index(true_answer) + 1
        return 1.0 / rank
    return 0.0

scores = []

for i in range(20):

    row = train.iloc[i]

    prompt = str(row["prompt"])
    true_letter = row["answer"]

    labels = {
        "A": str(row["A"]),
        "B": str(row["B"]),
        "C": str(row["C"]),
        "D": str(row["D"]),
        "E": str(row["E"])
    }

    # --------------------------------------------------
    # STEP 1: Retrieve top-5 documents using FAISS
    # --------------------------------------------------
    query_embedding = model.encode([prompt])

    distances, indices = index.search(query_embedding, k=5)

    retrieved_docs = [kb[idx] for idx in indices[0]]

    # --------------------------------------------------
    # STEP 2: Rerank using Cross-Encoder
    # --------------------------------------------------
    pairs = [[prompt, doc] for doc in retrieved_docs]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = retrieved_docs[np.argmax(ce_scores)]

    # --------------------------------------------------
    # STEP 3: Create RAG prompt
    # --------------------------------------------------
    rag_prompt = f"Context: {best_doc} Question: {prompt}"

    # --------------------------------------------------
    # STEP 4: Zero-shot prediction
    # --------------------------------------------------
    candidate_labels = list(labels.values())

    result = zs(
        rag_prompt,
        candidate_labels=candidate_labels,
        multi_label=False
    )

    # --------------------------------------------------
    # STEP 5: Convert predicted texts back to option letters
    # --------------------------------------------------
    text_to_letter = {v: k for k, v in labels.items()}

    predicted_letters = [
        text_to_letter[label]
        for label in result["labels"]
    ]

    top3 = predicted_letters[:3]

    score = map_at_3(top3, true_letter)

    scores.append(score)

    print(
        f"Row {i:2d} | True={true_letter} | "
        f"Top3={top3} | MAP@3={score:.3f}"
    )

# ------------------------------------------------------
# Final Average MAP@3
# ------------------------------------------------------
final_map3 = np.mean(scores)

print("\n===================================")
print(f"Average MAP@3 = {final_map3:.3f}")
print("===================================")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Row  0 | True=B | Top3=['B', 'D', 'A'] | MAP@3=1.000
Row  1 | True=A | Top3=['A', 'E', 'C'] | MAP@3=1.000
Row  2 | True=C | Top3=['C', 'D', 'B'] | MAP@3=1.000
Row  3 | True=B | Top3=['B', 'D', 'A'] | MAP@3=1.000
Row  4 | True=A | Top3=['A', 'B', 'C'] | MAP@3=1.000
Row  5 | True=C | Top3=['B', 'C', 'A'] | MAP@3=0.500
Row  6 | True=E | Top3=['E', 'B', 'D'] | MAP@3=1.000
Row  7 | True=A | Top3=['A', 'B', 'C'] | MAP@3=1.000
Row  8 | True=A | Top3=['A', 'C', 'D'] | MAP@3=1.000
Row  9 | True=A | Top3=['A', 'B', 'C'] | MAP@3=1.000
Row 10 | True=C | Top3=['C', 'A', 'D'] | MAP@3=1.000
Row 11 | True=B | Top3=['B', 'C', 'A'] | MAP@3=1.000
Row 12 | True=D | Top3=['D', 'A', 'B'] | MAP@3=1.000
Row 13 | True=E | Top3=['E', 'D', 'B'] | MAP@3=1.000
Row 14 | True=E | Top3=['E', 'A', 'D'] | MAP@3=1.000
Row 15 | True=E | Top3=['E', 'B', 'C'] | MAP@3=1.000
Row 16 | True=C | Top3=['C', 'B', 'E'] | MAP@3=1.000
Row 17 | True=C | Top3=['C', 'B', 'E'] | MAP@3=1.000
Row 18 | True=B | Top3=['B', 'D', 'A'] | MAP@3

In [ ]:
# 0.384
# 10
# 1
# 216
# 0.989
# 0.529
# 73
# 0.975